# 03 ACS — Drizzle and Inspect

**Inputs:** FLC/FLT files in `data_acs/raw/<target>/` downloaded by `01_download_data_acs.ipynb`.

All drizzle parameters read from `config/wfc3_ir_drizzle_params.yaml`.

In [ ]:
import os
import yaml
from pathlib import Path
from astropy.io import fits
from astropy import wcs
from astropy.visualization import ImageNormalize, LogStretch
import matplotlib.pyplot as plt
from drizzlepac import astrodrizzle
from IPython.display import clear_output

import subprocess
REPO_ROOT = Path(subprocess.check_output(['git', 'rev-parse', '--show-toplevel']).decode().strip())
RAW_DIR  = REPO_ROOT / 'data_acs/raw'
PROC_DIR = REPO_ROOT / 'data_acs/processed'
CONFIG   = REPO_ROOT / 'config' / 'wfc3_ir_drizzle_params.yaml'

In [ ]:
# Load drizzle parameters from config — nothing hardcoded here
drizzle_cfg = yaml.safe_load(open(CONFIG))['acs_drizzle']

# AstroDrizzle lowercases the output name when constructing intermediate filenames
# (e.g. static masks). Passing a path with mixed-case directory components causes
# a FileNotFoundError. Fix: cd into the output directory first and pass only the
# base filename — all intermediate and final files are then created in the right place.
original_dir = os.getcwd()

for quasar_dir in sorted(RAW_DIR.iterdir()):
    fl_files = sorted(quasar_dir.glob('*fl?.fits'))
    if not fl_files:
        continue

    target = quasar_dir.name
    print(f'\n{target}')

    # ACS uses FILTER1 and FILTER2 (two filter wheels) — pick the non-CLEAR filter
    by_filter = {}
    for fl in fl_files:
        f1 = fits.getval(str(fl), 'FILTER1', ext=0)
        f2 = fits.getval(str(fl), 'FILTER2', ext=0)
        filt = f1 if 'CLEAR' not in f1 else f2
        by_filter.setdefault(filt, []).append(str(fl.resolve()))  # absolute paths for safety

    for filt, files in sorted(by_filter.items()):
        out_dir = (PROC_DIR / target / 'drizzled').resolve()
        out_dir.mkdir(parents=True, exist_ok=True)

        # Change into the output directory so AstroDrizzle writes all files there
        os.chdir(out_dir)

        output_name = f'{target}_{filt}'
        print(f'  Drizzling {filt} ({len(files)} files) → {out_dir / output_name}')

        astrodrizzle.AstroDrizzle(
            files,
            output=output_name,
            context=False,       # skip context image — not needed for science
            build=True,          # build combined multi-extension FITS output
            preserve=False,      # do not keep intermediate drizzle files
            skymethod=drizzle_cfg['skymethod'],
            driz_separate=True,  # CR-rejection on — ACS CCDs require it (unlike WFC3/IR)
            median=True,         # CR-rejection on
            blot=True,           # CR-rejection on
            driz_cr=True,        # CR-rejection on
            final_wcs=drizzle_cfg['final_wcs'],
            final_rot=0.,        # orient output north-up
            final_scale=drizzle_cfg['final_scale'],
            final_pixfrac=drizzle_cfg['final_pixfrac'],
            final_kernel=drizzle_cfg['final_kernel'],
            driz_sep_bits=drizzle_cfg['driz_sep_bits'],
            final_bits=drizzle_cfg['final_bits'],
            combine_type=drizzle_cfg['combine_type'],
        )

        # Return to original directory before next iteration
        os.chdir(original_dir)

        # Clear verbose drizzle log output — completion message prints below
        clear_output(wait=True)
        print(f'  {target} {filt} complete')

print('\nAll quasars drizzled.')


In [ ]:
# Inspect drizzled science and weight images for all quasars and filters
# vmin/vmax control the display stretch — adjust per target if needed
vmin, vmax = -0.05, 100
GLIKMAN_DIR = Path('/home/daltshuler/RedQuasarResearch/GlikmanFIles')

for quasar_dir in sorted(PROC_DIR.iterdir()):
    drz_dir = quasar_dir / 'drizzled'
    drz_files = sorted(drz_dir.glob('*_drc.fits')) if drz_dir.exists() else []
    if not drz_files:
        continue

    target = quasar_dir.name

    for drz_file in drz_files:
        # Parse the filter name from the filename: <TARGET>_<FILTER>_drz.fits
        filt = drz_file.stem.replace(f'{target}_', '').replace('_drc', '')

        with fits.open(drz_file) as hdu:
            # ext=1 is the drizzled science image; ext=2 is the weight map
            im_wcs = wcs.WCS(hdu[1].header)
            sci    = hdu[1].data
            wht    = hdu[2].data

        # Log stretch makes faint extended emission visible alongside the bright core
        norm = ImageNormalize(sci, vmin=vmin, vmax=vmax, stretch=LogStretch())

        fig, axes = plt.subplots(1, 2, figsize=(20, 15), subplot_kw={'projection': im_wcs})
        fig.suptitle(f'{target} — {filt}', fontsize=20)

        # Left: science image with log stretch
        axes[0].imshow(sci, norm=norm, cmap='gray', origin='lower')
        axes[0].set_title('Science (SCI)', fontsize=16)

        # Right: weight map shows coverage — uniform weight = good dither fill
        axes[1].imshow(wht, cmap='gray', origin='lower')
        axes[1].set_title('Weight (WHT)', fontsize=16)

        for ax in axes:
            ax.tick_params(labelsize=13)
        plt.tight_layout()
        fig.savefig(drz_dir / f'{target}_{filt}_drc_inspect.png', dpi=150, bbox_inches='tight')
        glik_out = GLIKMAN_DIR / target
        glik_out.mkdir(parents=True, exist_ok=True)
        fig.savefig(glik_out / f'{target}_{filt}_drc_inspect.png', dpi=150, bbox_inches='tight')
        plt.show()